# Detection Pipeline (PKU-MMD / TSU) — Lightning AI

**Chiến lược I/O tối ưu cho đa luồng:**
- **Đọc video/annotation:** Streaming trực tiếp qua `rclone mount` từ Google Drive (không tốn local disk)
- **Ghi output (jsonl, failed_frames):** Ra local SSD của Lightning AI (I/O nhanh nhất, không nghẽn network)
- **Sync kết quả:** Dùng `rclone copy` gom lên Drive 1 lần khi xong

**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 (sync)

In [ ]:
# Cell 1: Lay code moi nhat tu GitHub
import os
REPO = "https://github.com/tuan8p/Skeleton-EAA-Pose.git"
BRANCH = "dai"
WORKDIR = "/teamspace/studios/this_studio/Skeleton-EAA-Pose"
if not os.path.isdir(WORKDIR):
    os.makedirs(os.path.dirname(WORKDIR), exist_ok=True)
    !git clone -b {BRANCH} {REPO} {WORKDIR}
else:
    !git -C {WORKDIR} pull origin {BRANCH}
%cd {WORKDIR}

In [ ]:
# Cell 2: Cai dat thu vien
!pip install -r requirements.txt

# Cai rclone neu chua co (tren Lightning AI thuong da co san)
!which rclone || (curl https://rclone.org/install.sh | bash)

In [ ]:
# Cell 3: Cau hinh rclone & Mount Google Drive
# ============================================================
# BUOC 1: Tao rclone config neu chua co
# Chay lenh nay 1 lan roi comment lai:
#   !rclone config
#   => Chon 'n' (new remote) -> ten: 'gdrive' -> chon 'drive'
#   => Paste client_id, client_secret (hoac bo trong de dung default)
#   => Scope: 1 (full access) -> Auto config: N -> paste Auth Token
#
# BUOC 2: Hoac dung headless auth (khuyen nghi cho Lightning AI):
#   !rclone authorize "drive"   # chay tren may tinh co browser, lay token
#   Paste token vao bien RCLONE_TOKEN duoi day
# ============================================================

import os

GDRIVE_REMOTE = "gdrive"   # ten remote da dat trong rclone config
MOUNT_POINT = "/teamspace/studios/this_studio/gdrive_mount"

os.makedirs(MOUNT_POINT, exist_ok=True)

# Kiem tra xem da mount chua
import subprocess
result = subprocess.run(["mountpoint", "-q", MOUNT_POINT])
if result.returncode != 0:
    # Mount Google Drive voi cac tuy chon toi uu cho doc file lon
    !rclone mount {GDRIVE_REMOTE}: {MOUNT_POINT} \
        --daemon \
        --vfs-cache-mode full \
        --vfs-cache-max-size 20G \
        --vfs-read-ahead 512M \
        --transfers 8 \
        --buffer-size 256M \
        --drive-chunk-size 128M \
        --log-level ERROR
    import time; time.sleep(5)   # cho rclone khoi dong
    print(f"Mounted Google Drive tai: {MOUNT_POINT}")
else:
    print(f"Da mount san tai: {MOUNT_POINT}")

# Kiem tra mount thanh cong
!ls {MOUNT_POINT} | head -5

In [ ]:
# Cell 4: Cau hinh - chon dataset va duong dan
DATASET = "PKU"   # "PKU" hoac "TSU"

# Goc cua Google Drive (sau diem mount)
DRIVE_ROOT = "/teamspace/studios/this_studio/gdrive_mount/MyDrive"

# =======================================================
# PATH CONFIG: chinh theo cay thu muc Google Drive cua ban
# =======================================================
PKU_PATHS = {
    "video_dir":      f"{DRIVE_ROOT}/ĐACN-TN_datasets/ĐATN/rawdatasets/PKUMMD/Data/RGB_VIDEO",
    "annotation_dir": f"{DRIVE_ROOT}/ĐACN-TN_datasets/ĐATN/rawdatasets/skeletons/PKU/Label_PKUMMD_v1",
}
TSU_PATHS = {
    "video_dir":      f"{DRIVE_ROOT}/ĐACN-TN_datasets/ĐATN/rawdatasets/videos/TSU",
    "annotation_dir": f"{DRIVE_ROOT}/ĐACN-TN_datasets/ĐATN/rawdatasets/skeletons/TSU/Annotation_v1.0",
}
DEPTH_PATHS = {
    "PKU": f"{DRIVE_ROOT}/PKU-MMD/DEPTH_PKUMMD",
    "TSU": f"{DRIVE_ROOT}/TSU/videos",
}

# Output ghi ra LOCAL SSD cua Lightning AI (I/O nhanh nhat)
# Cuoi session se sync len Drive (Cell 8)
LOCAL_OUTPUT = "/teamspace/studios/this_studio/outputs_detection"

# Dich tren Drive de sync output len sau
DRIVE_OUTPUT = f"{DRIVE_ROOT}/ĐACN-TN_datasets/ĐATN/rawdatasets/outputs_detection"

# ==========================================
# CAU HINH PIPELINE (Chinh truc tiep o day)
# ==========================================
SETTINGS = {
    "yolo.model_name": "yolo26x.pt",          # yolo26n.pt (nhe) | yolo26l.pt | yolo26x.pt (nang)
    "yolo.tracker": "iou",                    # iou: nhe, khong mat frame; deepocsort: nang hon
    "runtime.num_workers": 4,                 # Song song 4 video (T4: 4 la max on dinh)
    "yolo.batch_size": 32,                    # T4 15GB: batch 32 la sweet spot
    "yolo.conf_threshold": 0.2,               # Nguong tin cay bbox
    "yolo.iou_threshold": 0.7,                # Nguong IoU NMS
    "detection.retry.max_retries": 2,         # So lan thu lai khi miss person
    "detection.retry.imgsz_retry": 1280,      # Retry 1: zoom in anh len 1280
    "detection.retry.tta": False,             # Test Time Augmentation (tat de giu VRAM)
    "detection.retry.clahe_on_retry": True,   # Retry 2: tang tuong phan CLAHE
    "chunk.max_actions_per_chunk": 5,         # So action toi da moi chunk
    # --- TSU Specific Config ---
    "tsu.video_ext": ".mp4",
    "tsu.has_header": True,
    "tsu.event_column": 0,
    "tsu.start_column": 1,
    "tsu.end_column": 2,
    "tsu.event_map": {},                      # {"event_name": action_id} | {} = giu nguyen
    "tsu.event_mapping_path": f"{DRIVE_ROOT}/ĐACN-TN_datasets/ĐATN/rawdatasets/event_mapping.csv",  # File anh xa event TSU co dinh
}

USE_DEPTH = False   # True = bat ghost legs masking (chi dung cho PKU)
START, END = 0, 1   # Slice danh sach video [START:END], vi du 0:10 = chay 10 video dau

In [ ]:
# Cell 5: Tao cac thu muc local output va kiem tra mount
import os

os.makedirs(LOCAL_OUTPUT, exist_ok=True)
print(f"Local output dir: {LOCAL_OUTPUT}")

# Kiem tra doc duoc video tu Drive chua
paths = PKU_PATHS if DATASET == "PKU" else TSU_PATHS
video_dir = paths["video_dir"]
ann_dir   = paths["annotation_dir"]

assert os.path.isdir(video_dir), f"Khong tim thay video_dir: {video_dir}"
assert os.path.isdir(ann_dir),   f"Khong tim thay annotation_dir: {ann_dir}"

videos = sorted(os.listdir(video_dir))
print(f"[{DATASET}] Tim thay {len(videos)} files trong video_dir")
print(f"3 file dau: {videos[:3]}")
print(f"[OK] Tat ca path hop le, san sang chay pipeline!")

In [ ]:
# Cell 6: Nap config, ghi de path va settings
from src.config_manager import ConfigManager

cfg = ConfigManager("config.yaml")
cfg.set("dataset", DATASET)

# Ghi paths
paths = PKU_PATHS if DATASET == "PKU" else TSU_PATHS
for k, v in paths.items():
    cfg.set(f"paths.{k}", v)

# Output ra LOCAL SSD (khong phai Drive) de I/O nhanh
cfg.set("detection.output_dir", LOCAL_OUTPUT)
cfg.set("paths.depth_dir", DEPTH_PATHS[DATASET])
cfg.set("depth.enabled", USE_DEPTH)

# Ghi tat ca SETTINGS tu Cell 4
for k, v in SETTINGS.items():
    cfg.set(k, v)

cfg.save("config.runtime.yaml")   # Luu lai de kiem tra / reproduce
print(cfg.as_dict())

In [ ]:
# Cell 7: Chay pipeline (co resume tu chunk/video cuoi neu bi ngat)
from src.detection_pipeline import DetectionPipeline

orch = DetectionPipeline(cfg=cfg)
orch.run_batch(start=START, end=END)

In [ ]:
# Cell 8: Sync output tu local SSD len Google Drive
# Chay sau khi pipeline hoan tat (hoac bat cu luc nao muon backup)
# rclone copy: chi copy file moi/thay doi (khong xoa file cu tren Drive)
import subprocess, datetime

print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Bat dau sync len Drive...")

result = subprocess.run([
    "rclone", "copy",
    LOCAL_OUTPUT,
    DRIVE_OUTPUT,
    "--transfers", "8",           # 8 luong upload song song
    "--drive-chunk-size", "128M", # chunk size lon de upload file to nhanh
    "--progress",
    "--log-level", "INFO"
], capture_output=False, text=True)

if result.returncode == 0:
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Sync thanh cong len: {DRIVE_OUTPUT}")
else:
    print(f"[LOI] rclone exit code: {result.returncode}")

## Ghi chu toi uu I/O cho Lightning AI

| Chien luoc | Ly do |
|---|---|
| **rclone mount** `--vfs-cache-mode full` | Cache video vao RAM/disk de `cv2.VideoCapture` doc streaming on dinh, tranh timeout |
| `--vfs-read-ahead 512M` | Pre-fetch du lieu video lien tuc truoc khi can doc, giam lat khi seek frame |
| `--buffer-size 256M` | Buffer truyen du lieu lon, giam so luot round-trip API |
| **Output ra local SSD** | Local NVMe SSD cua Lightning AI co IOPS ~500,000 so voi Drive API ~100 req/s: ghi nhanh hon ~50x khi nhieu luong ghi dong thoi |
| **Sync 1 lan cuoi** | Tranh nghen co chai (bottleneck) khi 4 worker cung ghi file jsonl truoc khi ra network |
| `--transfers 8` khi sync | Upload song song 8 file, giam thoi gian sync tong the |